# 🎯 Mustererkennung in Produktionsdaten

**Intelligente Produktionsplanung und KI**  
Universität Stuttgart  
Master Maschinenbau / Technologiemanagement

---

In diesem Notebook analysieren wir Produktions- und Qualitätsdaten mit Methoden der **Mustererkennung**:
- Clustering von Produkten (K-Means)
- Visualisierung und Interpretation von Clustern
- Anomalieerkennung in Qualitätsdaten (Isolation Forest, Z-Score)
- Bewertung und Visualisierung der Anomalieerkennung


In [ ]:
# ===================================================================
# Block 3.2: Mustererkennung in Produktionsdaten
# Intelligente Produktionsplanung und KI
# Universität Stuttgart
# ===================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.metrics import silhouette_score
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Deutsche Plots
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11
sns.set_style("whitegrid")

print("🎯 Mustererkennung in Produktionsdaten")
print("=" * 45)

In [ ]:
# ===================================================================
# 1. DATENSATZ GENERIEREN: PRODUKTIONSDATEN
# ===================================================================

def generiere_produktionsdaten(n_produkte=200, seed=42):
    """
    Generiert realistische Produktionsdaten für Clustering-Analyse
    """
    np.random.seed(seed)
    
    # Produktkategorien definieren
    kategorien = {
        'Elektronik': {'basis_kosten': 150, 'basis_zeit': 45, 'basis_qualitaet': 0.95},
        'Mechanik': {'basis_kosten': 80, 'basis_zeit': 120, 'basis_qualitaet': 0.92},
        'Optik': {'basis_kosten': 300, 'basis_zeit': 180, 'basis_qualitaet': 0.98},
        'Software': {'basis_kosten': 50, 'basis_zeit': 30, 'basis_qualitaet': 0.90}
    }
    
    daten = []
    
    for i in range(n_produkte):
        # Zufällige Kategorie wählen
        kategorie = np.random.choice(list(kategorien.keys()))
        basis = kategorien[kategorie]
        
        # Produkteigenschaften mit Variation generieren
        produkt = {
            'Produkt_ID': f'P{i+1:03d}',
            'Kategorie': kategorie,
            'Materialkosten': np.random.normal(basis['basis_kosten'], basis['basis_kosten']*0.2),
            'Fertigungszeit_min': np.random.normal(basis['basis_zeit'], basis['basis_zeit']*0.3),
            'Qualitaetsscore': np.random.beta(basis['basis_qualitaet']*20, (1-basis['basis_qualitaet'])*20),
            'Komplexitaet': np.random.uniform(1, 10),
            'Ausschussrate': np.random.exponential(0.05),
            'Energieverbrauch_kWh': np.random.gamma(2, 5),
            'Gewicht_kg': np.random.lognormal(1, 0.5),
            'Volumen_cm3': np.random.lognormal(4, 0.8)
        }
        
        # Korrelationen einbauen
        if kategorie == 'Elektronik':
            produkt['Energieverbrauch_kWh'] *= 0.5  # Elektronik verbraucht weniger Energie
            produkt['Gewicht_kg'] *= 0.3  # Leichter
        elif kategorie == 'Mechanik':
            produkt['Gewicht_kg'] *= 2.0  # Schwerer
            produkt['Fertigungszeit_min'] *= 1.2  # Länger
        elif kategorie == 'Optik':
            produkt['Qualitaetsscore'] = min(produkt['Qualitaetsscore'] * 1.05, 1.0)  # Höhere Qualität
            produkt['Komplexitaet'] *= 1.3  # Komplexer
        
        # Negative Werte vermeiden
        for key in ['Materialkosten', 'Fertigungszeit_min', 'Energieverbrauch_kWh', 'Gewicht_kg', 'Volumen_cm3']:
            produkt[key] = max(produkt[key], 0.1)
        
        produkt['Qualitaetsscore'] = np.clip(produkt['Qualitaetsscore'], 0.7, 1.0)
        produkt['Ausschussrate'] = np.clip(produkt['Ausschussrate'], 0.001, 0.2)
        
        daten.append(produkt)
    
    return pd.DataFrame(daten)

# Produktionsdaten generieren
produktions_df = generiere_produktionsdaten()

print("📊 GENERIERTE PRODUKTIONSDATEN")
print("-" * 35)
print(f"Anzahl Produkte: {len(produktions_df)}")
print(f"Kategorien: {produktions_df['Kategorie'].unique()}")
print(f"\nErste 5 Datenpunkte:")
print(produktions_df.head())

print(f"\nStatistische Übersicht:")
numerische_spalten = produktions_df.select_dtypes(include=[np.number]).columns
print(produktions_df[numerische_spalten].describe().round(2))

In [ ]:
# ===================================================================
# 2. EXPLORATIVE DATENANALYSE
# ===================================================================

def explorative_analyse(df):
    """Führt explorative Datenanalyse durch"""
    
    print("\n🔍 EXPLORATIVE DATENANALYSE")
    print("-" * 30)
    
    # Kategorienverteilung
    plt.figure(figsize=(15, 12))
    
    # 1. Kategorienverteilung
    plt.subplot(2, 3, 1)
    kategorie_counts = df['Kategorie'].value_counts()
    plt.pie(kategorie_counts.values, labels=kategorie_counts.index, autopct='%1.1f%%')
    plt.title('Verteilung der Produktkategorien')
    
    # 2. Kosten vs. Zeit
    plt.subplot(2, 3, 2)
    for kategorie in df['Kategorie'].unique():
        subset = df[df['Kategorie'] == kategorie]
        plt.scatter(subset['Materialkosten'], subset['Fertigungszeit_min'], 
                   label=kategorie, alpha=0.7)
    plt.xlabel('Materialkosten (€)')
    plt.ylabel('Fertigungszeit (min)')
    plt.title('Kosten vs. Fertigungszeit')
    plt.legend()
    
    # 3. Qualität vs. Ausschuss
    plt.subplot(2, 3, 3)
    plt.scatter(df['Qualitaetsscore'], df['Ausschussrate'], alpha=0.6)
    plt.xlabel('Qualitätsscore')
    plt.ylabel('Ausschussrate')
    plt.title('Qualität vs. Ausschussrate')
    
    # 4. Komplexität Verteilung
    plt.subplot(2, 3, 4)
    df['Komplexitaet'].hist(bins=20, alpha=0.7)
    plt.xlabel('Komplexität')
    plt.ylabel('Häufigkeit')
    plt.title('Verteilung der Komplexität')
    
    # 5. Energieverbrauch nach Kategorie
    plt.subplot(2, 3, 5)
    df.boxplot(column='Energieverbrauch_kWh', by='Kategorie', ax=plt.gca())
    plt.title('Energieverbrauch nach Kategorie')
    plt.suptitle('')  # Entfernt automatischen Titel
    
    # 6. Korrelationsmatrix
    plt.subplot(2, 3, 6)
    korr_matrix = df[numerische_spalten].corr()
    sns.heatmap(korr_matrix, annot=True, cmap='coolwarm', center=0, 
                square=True, fmt='.2f', cbar_kws={'shrink': 0.8})
    plt.title('Korrelationsmatrix')
    
    plt.tight_layout()
    plt.show()
    
    # Statistische Zusammenfassung nach Kategorie
    print("\nStatistiken nach Kategorie:")
    for kategorie in df['Kategorie'].unique():
        subset = df[df['Kategorie'] == kategorie]
        print(f"\n{kategorie}:")
        print(f"  Anzahl: {len(subset)}")
        print(f"  Ø Materialkosten: {subset['Materialkosten'].mean():.1f} €")
        print(f"  Ø Fertigungszeit: {subset['Fertigungszeit_min'].mean():.1f} min")
        print(f"  Ø Qualitätsscore: {subset['Qualitaetsscore'].mean():.3f}")
        print(f"  Ø Ausschussrate: {subset['Ausschussrate'].mean():.3f}")

explorative_analyse(produktions_df)

In [ ]:
# ===================================================================
# 3. K-MEANS CLUSTERING
# ===================================================================

class ProduktClustering:
    """Klasse für Produkt-Clustering-Analysen"""
    
    def __init__(self, df):
        self.df = df.copy()
        self.features = ['Materialkosten', 'Fertigungszeit_min', 'Qualitaetsscore', 
                        'Komplexitaet', 'Ausschussrate', 'Energieverbrauch_kWh']
        self.scaler = StandardScaler()
        self.X_scaled = None
        self.cluster_models = {}
    
    def vorbereitung_daten(self):
        """Bereitet Daten für Clustering vor"""
        # Nur numerische Features für Clustering
        X = self.df[self.features].copy()
        
        # Skalierung
        self.X_scaled = self.scaler.fit_transform(X)
        
        print("📋 DATEN-VORBEREITUNG")
        print("-" * 25)
        print(f"Features für Clustering: {self.features}")
        print(f"Datenform nach Skalierung: {self.X_scaled.shape}")
        
        return self.X_scaled
    
    def optimale_cluster_anzahl(self, max_k=10):
        """Bestimmt optimale Anzahl Cluster mit Elbow-Methode und Silhouette-Score"""
        
        if self.X_scaled is None:
            self.vorbereitung_daten()
        
        print("\n🔍 OPTIMALE CLUSTER-ANZAHL BESTIMMEN")
        print("-" * 40)
        
        # Verschiedene k-Werte testen
        k_range = range(2, max_k + 1)
        inertias = []
        silhouette_scores = []
        
        for k in k_range:
            kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
            kmeans.fit(self.X_scaled)
            
            inertias.append(kmeans.inertia_)
            sil_score = silhouette_score(self.X_scaled, kmeans.labels_)
            silhouette_scores.append(sil_score)
            
            print(f"k={k}: Inertia={kmeans.inertia_:.2f}, Silhouette={sil_score:.3f}")
        
        # Visualisierung
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
        
        # Elbow-Kurve
        ax1.plot(k_range, inertias, 'bo-', linewidth=2, markersize=8)
        ax1.set_xlabel('Anzahl Cluster (k)')
        ax1.set_ylabel('Inertia (Within-Cluster Sum of Squares)')
        ax1.set_title('Elbow-Methode')
        ax1.grid(True, alpha=0.3)
        
        # Silhouette-Scores
        ax2.plot(k_range, silhouette_scores, 'ro-', linewidth=2, markersize=8)
        ax2.set_xlabel('Anzahl Cluster (k)')
        ax2.set_ylabel('Silhouette Score')
        ax2.set_title('Silhouette-Analyse')
        ax2.grid(True, alpha=0.3)
        
        # Optimum markieren
        best_k_silhouette = k_range[np.argmax(silhouette_scores)]
        ax2.axvline(x=best_k_silhouette, color='red', linestyle='--', 
                   label=f'Optimum: k={best_k_silhouette}')
        ax2.legend()
        
        plt.tight_layout()
        plt.show()
        
        print(f"\n🏆 Empfohlene Cluster-Anzahl: {best_k_silhouette} (höchster Silhouette-Score)")
        
        return best_k_silhouette
    
    def kmeans_clustering(self, n_clusters=4):
        """Führt K-Means Clustering durch"""
        
        if self.X_scaled is None:
            self.vorbereitung_daten()
        
        print(f"\n🎯 K-MEANS CLUSTERING (k={n_clusters})")
        print("-" * 35)
        
        # K-Means anwenden
        kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        cluster_labels = kmeans.fit_predict(self.X_scaled)
        
        # Ergebnisse zum DataFrame hinzufügen
        self.df['Cluster'] = cluster_labels
        self.cluster_models['kmeans'] = kmeans
        
        # Cluster-Statistiken
        print("Cluster-Verteilung:")
        cluster_counts = pd.Series(cluster_labels).value_counts().sort_index()
        for cluster_id, count in cluster_counts.items():
            print(f"  Cluster {cluster_id}: {count} Produkte ({count/len(self.df)*100:.1f}%)")
        
        # Silhouette Score
        sil_score = silhouette_score(self.X_scaled, cluster_labels)
        print(f"\nSilhouette Score: {sil_score:.3f}")
        
        return cluster_labels
    
    def cluster_charakterisierung(self):
        """Charakterisiert die gefundenen Cluster"""
        
        print("\n📊 CLUSTER-CHARAKTERISIERUNG")
        print("-" * 35)
        
        # Durchschnittswerte pro Cluster
        cluster_stats = self.df.groupby('Cluster')[self.features].mean()
        
        print("Durchschnittswerte pro Cluster:")
        print(cluster_stats.round(2))
        
        # Cluster-Namen basierend auf Charakteristika vergeben
        cluster_namen = {}
        for cluster_id in cluster_stats.index:
            stats = cluster_stats.loc[cluster_id]
            
            if stats['Materialkosten'] > cluster_stats['Materialkosten'].mean():
                if stats['Qualitaetsscore'] > cluster_stats['Qualitaetsscore'].mean():
                    cluster_namen[cluster_id] = "Premium-Produkte"
                else:
                    cluster_namen[cluster_id] = "Teure Standardprodukte"
            else:
                if stats['Fertigungszeit_min'] < cluster_stats['Fertigungszeit_min'].mean():
                    cluster_namen[cluster_id] = "Schnelle Produktion"
                else:
                    cluster_namen[cluster_id] = "Kostengünstige Produkte"
        
        self.df['Cluster_Name'] = self.df['Cluster'].map(cluster_namen)
        
        print(f"\nCluster-Bezeichnungen:")
        for cluster_id, name in cluster_namen.items():
            print(f"  Cluster {cluster_id}: {name}")
        
        return cluster_namen
    
    def visualisiere_cluster(self):
        """Visualisiert die Cluster-Ergebnisse"""
        
        print("\n📈 CLUSTER-VISUALISIERUNG")
        print("-" * 30)
        
        # PCA für 2D-Visualisierung
        pca = PCA(n_components=2)
        X_pca = pca.fit_transform(self.X_scaled)
        
        # Erklärte Varianz
        explained_var = pca.explained_variance_ratio_
        print(f"Erklärte Varianz durch erste 2 Komponenten: {sum(explained_var):.1%}")
        
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        
        # 1. PCA-Visualisierung
        scatter = axes[0, 0].scatter(X_pca[:, 0], X_pca[:, 1], 
                                   c=self.df['Cluster'], cmap='viridis', alpha=0.7)
        axes[0, 0].set_xlabel(f'PC1 ({explained_var[0]:.1%} Varianz)')
        axes[0, 0].set_ylabel(f'PC2 ({explained_var[1]:.1%} Varianz)')
        axes[0, 0].set_title('Cluster in PCA-Raum')
        plt.colorbar(scatter, ax=axes[0, 0])
        
        # 2. Kosten vs. Zeit (gefärbt nach Cluster)
        for cluster_id in self.df['Cluster'].unique():
            subset = self.df[self.df['Cluster'] == cluster_id]
            axes[0, 1].scatter(subset['Materialkosten'], subset['Fertigungszeit_min'],
                             label=f'Cluster {cluster_id}', alpha=0.7)
        axes[0, 1].set_xlabel('Materialkosten (€)')
        axes[0, 1].set_ylabel('Fertigungszeit (min)')
        axes[0, 1].set_title('Kosten vs. Zeit nach Clustern')
        axes[0, 1].legend()
        
        # 3. Qualität vs. Ausschuss (gefärbt nach Cluster)
        for cluster_id in self.df['Cluster'].unique():
            subset = self.df[self.df['Cluster'] == cluster_id]
            axes[1, 0].scatter(subset['Qualitaetsscore'], subset['Ausschussrate'],
                             label=f'Cluster {cluster_id}', alpha=0.7)
        axes[1, 0].set_xlabel('Qualitätsscore')
        axes[1, 0].set_ylabel('Ausschussrate')
        axes[1, 0].set_title('Qualität vs. Ausschuss nach Clustern')
        axes[1, 0].legend()
        
        # 4. Cluster-Größen
        cluster_counts = self.df['Cluster'].value_counts().sort_index()
        axes[1, 1].bar(cluster_counts.index, cluster_counts.values, alpha=0.7)
        axes[1, 1].set_xlabel('Cluster')
        axes[1, 1].set_ylabel('Anzahl Produkte')
        axes[1, 1].set_title('Cluster-Größen')
        
        plt.tight_layout()
        plt.show()

# Clustering durchführen
clusterer = ProduktClustering(produktions_df)
optimale_k = clusterer.optimale_cluster_anzahl()
cluster_labels = clusterer.kmeans_clustering(n_clusters=optimale_k)
cluster_namen = clusterer.cluster_charakterisierung()
clusterer.visualisiere_cluster()

In [ ]:
# ===================================================================
# 4. ANOMALIEERKENNUNG IN QUALITÄTSDATEN
# ===================================================================

def generiere_qualitaetsdaten(n_messungen=1000, seed=42):
    """Generiert Qualitätsmessungen mit Anomalien"""
    
    np.random.seed(seed)
    
    # Normale Qualitätsmessungen
    normale_daten = []
    
    for i in range(int(n_messungen * 0.95)):  # 95% normale Daten
        messung = {
            'Messung_ID': f'M{i+1:04d}',
            'Temperatur_C': np.random.normal(22, 2),
            'Feuchtigkeit_Prozent': np.random.normal(45, 5),
            'Druck_bar': np.random.normal(1.013, 0.05),
            'Vibration_mm_s': np.random.exponential(0.5),
            'Oberflaechenrauheit_um': np.random.gamma(2, 0.5),
            'Durchmesser_mm': np.random.normal(10.0, 0.1),
            'Laenge_mm': np.random.normal(50.0, 0.5),
            'Gewicht_g': np.random.normal(25.0, 1.0),
            'Typ': 'Normal'
        }
        normale_daten.append(messung)
    
    # Anomalien hinzufügen
    anomalien = []
    n_anomalien = n_messungen - len(normale_daten)
    
    for i in range(n_anomalien):
        # Verschiedene Arten von Anomalien
        anomalie_typ = np.random.choice(['Temperatur', 'Dimension', 'Vibration', 'Multi'])
        
        if anomalie_typ == 'Temperatur':
            messung = {
                'Messung_ID': f'A{i+1:04d}',
                'Temperatur_C': np.random.choice([np.random.normal(35, 2), np.random.normal(5, 2)]),
                'Feuchtigkeit_Prozent': np.random.normal(45, 5),
                'Druck_bar': np.random.normal(1.013, 0.05),
                'Vibration_mm_s': np.random.exponential(0.5),
                'Oberflaechenrauheit_um': np.random.gamma(2, 0.5),
                'Durchmesser_mm': np.random.normal(10.0, 0.1),
                'Laenge_mm': np.random.normal(50.0, 0.5),
                'Gewicht_g': np.random.normal(25.0, 1.0),
                'Typ': 'Anomalie_Temperatur'
            }
        elif anomalie_typ == 'Dimension':
            messung = {
                'Messung_ID': f'A{i+1:04d}',
                'Temperatur_C': np.random.normal(22, 2),
                'Feuchtigkeit_Prozent': np.random.normal(45, 5),
                'Druck_bar': np.random.normal(1.013, 0.05),
                'Vibration_mm_s': np.random.exponential(0.5),
                'Oberflaechenrauheit_um': np.random.gamma(2, 0.5),
                'Durchmesser_mm': np.random.normal(10.0, 0.5),  # Größere Variation
                'Laenge_mm': np.random.normal(50.0, 2.0),       # Größere Variation
                'Gewicht_g': np.random.normal(25.0, 3.0),       # Größere Variation
                'Typ': 'Anomalie_Dimension'
            }
        elif anomalie_typ == 'Vibration':
            messung = {
                'Messung_ID': f'A{i+1:04d}',
                'Temperatur_C': np.random.normal(22, 2),
                'Feuchtigkeit_Prozent': np.random.normal(45, 5),
                'Druck_bar': np.random.normal(1.013, 0.05),
                'Vibration_mm_s': np.random.exponential(3.0),  # Viel höhere Vibration
                'Oberflaechenrauheit_um': np.random.gamma(5, 1),  # Schlechtere Oberfläche
                'Durchmesser_mm': np.random.normal(10.0, 0.1),
                'Laenge_mm': np.random.normal(50.0, 0.5),
                'Gewicht_g': np.random.normal(25.0, 1.0),
                'Typ': 'Anomalie_Vibration'
            }
        else:  # Multi-Anomalie
            messung = {
                'Messung_ID': f'A{i+1:04d}',
                'Temperatur_C': np.random.normal(30, 3),
                'Feuchtigkeit_Prozent': np.random.normal(70, 10),
                'Druck_bar': np.random.normal(0.9, 0.1),
                'Vibration_mm_s': np.random.exponential(2.0),
                'Oberflaechenrauheit_um': np.random.gamma(4, 1),
                'Durchmesser_mm': np.random.normal(10.0, 0.3),
                'Laenge_mm': np.random.normal(50.0, 1.5),
                'Gewicht_g': np.random.normal(25.0, 2.0),
                'Typ': 'Anomalie_Multi'
            }
        
        anomalien.append(messung)
    
    # Alle Daten kombinieren
    alle_daten = normale_daten + anomalien
    np.random.shuffle(alle_daten)  # Mischen
    
    return pd.DataFrame(alle_daten)

# Qualitätsdaten generieren
qualitaets_df = generiere_qualitaetsdaten()

print("\n🔍 QUALITÄTSDATEN FÜR ANOMALIEERKENNUNG")
print("-" * 45)
print(f"Anzahl Messungen: {len(qualitaets_df)}")
print(f"Anomalie-Typen: {qualitaets_df['Typ'].value_counts()}")

In [ ]:
# ===================================================================
# 5. ANOMALIEERKENNUNG IMPLEMENTIEREN
# ===================================================================

class AnomalieErkennung:
    """Klasse für Anomalieerkennung in Qualitätsdaten"""
    
    def __init__(self, df):
        self.df = df.copy()
        self.features = ['Temperatur_C', 'Feuchtigkeit_Prozent', 'Druck_bar', 
                        'Vibration_mm_s', 'Oberflaechenrauheit_um', 
                        'Durchmesser_mm', 'Laenge_mm', 'Gewicht_g']
        self.scaler = StandardScaler()
        self.X_scaled = None
        self.anomalie_modelle = {}
    
    def vorbereitung_daten(self):
        """Bereitet Daten für Anomalieerkennung vor"""
        X = self.df[self.features].copy()
        self.X_scaled = self.scaler.fit_transform(X)
        
        print("📋 DATEN-VORBEREITUNG ANOMALIEERKENNUNG")
        print("-" * 40)
        print(f"Features: {self.features}")
        print(f"Datenform: {self.X_scaled.shape}")
        
        return self.X_scaled
    
    def isolation_forest_anomalien(self, contamination=0.1):
        """Erkennt Anomalien mit Isolation Forest"""
        
        if self.X_scaled is None:
            self.vorbereitung_daten()
        
        print(f"\n🌲 ISOLATION FOREST (Kontamination: {contamination})")
        print("-" * 45)
        
        # Isolation Forest anwenden
        iso_forest = IsolationForest(contamination=contamination, random_state=42)
        anomalie_labels = iso_forest.fit_predict(self.X_scaled)
        
        # -1 = Anomalie, 1 = Normal
        self.df['IF_Anomalie'] = (anomalie_labels == -1)
        self.anomalie_modelle['isolation_forest'] = iso_forest
        
        # Ergebnisse
        n_anomalien = sum(anomalie_labels == -1)
        print(f"Erkannte Anomalien: {n_anomalien} ({n_anomalien/len(self.df)*100:.1f}%)")
        
        return anomalie_labels
    
    def statistische_anomalien(self, z_threshold=3):
        """Erkennt Anomalien mit Z-Score Methode"""
        
        print(f"\n📊 STATISTISCHE ANOMALIEERKENNUNG (Z-Score > {z_threshold})")
        print("-" * 55)
        
        # Z-Scores berechnen
        z_scores = np.abs(stats.zscore(self.df[self.features]))
        
        # Anomalien: Mindestens ein Feature mit Z-Score > Threshold
        stat_anomalien = (z_scores > z_threshold).any(axis=1)
        self.df['Stat_Anomalie'] = stat_anomalien
        
        n_stat_anomalien = sum(stat_anomalien)
        print(f"Erkannte Anomalien: {n_stat_anomalien} ({n_stat_anomalien/len(self.df)*100:.1f}%)")
        
        return stat_anomalien
    
    def bewerte_anomalieerkennung(self):
        """Bewertet die Güte der Anomalieerkennung"""
        
        print("\n📈 BEWERTUNG DER ANOMALIEERKENNUNG")
        print("-" * 40)
        
        # Ground Truth: Alle nicht-normalen Typen sind Anomalien
        true_anomalien = self.df['Typ'] != 'Normal'
        
        # Bewertung für beide Methoden
        methoden = {
            'Isolation Forest': 'IF_Anomalie',
            'Statistisch (Z-Score)': 'Stat_Anomalie'
        }
        
        for methode, spalte in methoden.items():
            if spalte in self.df.columns:
                predicted = self.df[spalte]
                
                # Confusion Matrix Werte
                tp = sum(true_anomalien & predicted)      # True Positives
                fp = sum(~true_anomalien & predicted)     # False Positives
                tn = sum(~true_anomalien & ~predicted)    # True Negatives
                fn = sum(true_anomalien & ~predicted)     # False Negatives
                
                # Metriken berechnen
                precision = tp / (tp + fp) if (tp + fp) > 0 else 0
                recall = tp / (tp + fn) if (tp + fn) > 0 else 0
                f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
                accuracy = (tp + tn) / len(self.df)
                
                print(f"\n{methode}:")
                print(f"  Precision: {precision:.3f}")
                print(f"  Recall: {recall:.3f}")
                print(f"  F1-Score: {f1_score:.3f}")
                print(f"  Accuracy: {accuracy:.3f}")
                print(f"  True Positives: {tp}")
                print(f"  False Positives: {fp}")
                print(f"  False Negatives: {fn}")
    
    def visualisiere_anomalien(self):
        """Visualisiert die erkannten Anomalien"""
        
        print("\n📊 ANOMALIE-VISUALISIERUNG")
        print("-" * 30)
        
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        axes = axes.ravel()
        
        # Feature-Paare für Visualisierung
        feature_paare = [
            ('Temperatur_C', 'Feuchtigkeit_Prozent'),
            ('Vibration_mm_s', 'Oberflaechenrauheit_um'),
            ('Durchmesser_mm', 'Laenge_mm'),
            ('Gewicht_g', 'Druck_bar'),
            ('Temperatur_C', 'Vibration_mm_s'),
            ('Durchmesser_mm', 'Gewicht_g')
        ]
        
        for i, (x_feature, y_feature) in enumerate(feature_paare):
            ax = axes[i]
            
            # Normale Punkte
            normale = self.df[self.df['Typ'] == 'Normal']
            ax.scatter(normale[x_feature], normale[y_feature], 
                      alpha=0.6, label='Normal', color='blue', s=20)
            
            # Echte Anomalien
            anomalien = self.df[self.df['Typ'] != 'Normal']
            ax.scatter(anomalien[x_feature], anomalien[y_feature], 
                      alpha=0.8, label='Echte Anomalie', color='red', s=30, marker='x')
            
            # Von Isolation Forest erkannte Anomalien
            if 'IF_Anomalie' in self.df.columns:
                if_anomalien = self.df[self.df['IF_Anomalie']]
                ax.scatter(if_anomalien[x_feature], if_anomalien[y_feature], 
                          alpha=0.5, label='IF erkannt', facecolors='none', 
                          edgecolors='orange', s=50, linewidth=2)
            
            ax.set_xlabel(x_feature)
            ax.set_ylabel(y_feature)
            ax.set_title(f'{x_feature} vs {y_feature}')
            ax.legend()
            ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Anomalie-Typen Analyse
        if 'IF_Anomalie' in self.df.columns:
            print("\nANOMALIE-TYPEN ANALYSE:")
            print("-" * 25)
            
            # Verwirrungsmatrix als Kreuztabelle
            confusion = pd.crosstab(self.df['Typ'], self.df['IF_Anomalie'], 
                                  margins=True, margins_name='Total')
            print("Isolation Forest Ergebnisse:")
            print(confusion)

# Anomalieerkennung durchführen
anomalie_detector = AnomalieErkennung(qualitaets_df)
if_labels = anomalie_detector.isolation_forest_anomalien(contamination=0.05)
stat_labels = anomalie_detector.statistische_anomalien(z_threshold=2.5)
anomalie_detector.bewerte_anomalieerkennung()
anomalie_detector.visualisiere_anomalien()

In [ ]:
# ===================================================================
# 6. AUFGABEN FÜR STUDIERENDE
# ===================================================================

print("\n✏️ AUFGABEN ZUM SELBST LÖSEN")
print("-" * 35)

print("""
AUFGABE 1: Cluster-Optimierung
Experimentieren Sie mit verschiedenen Clustering-Parametern:
- Testen Sie DBSCAN als Alternative zu K-Means
- Variieren Sie die Features für das Clustering
- Interpretieren Sie die Unterschiede in den Ergebnissen

AUFGABE 2: Anomalie-Parameter-Tuning
Optimieren Sie die Anomalieerkennung:
- Testen Sie verschiedene Contamination-Werte (0.01, 0.05, 0.1, 0.15)
- Variieren Sie den Z-Score Threshold (2, 2.5, 3, 3.5)
- Welche Kombination gibt die besten F1-Scores?

AUFGABE 3: Feature Engineering
Erstellen Sie neue Features für bessere Mustererkennung:
- Verhältnisse zwischen Features (z.B. Kosten/Zeit)
- Kategorische Features in numerische umwandeln
- Testen Sie den Einfluss auf Clustering-Qualität

AUFGABE 4: Praktische Anwendung
Entwickeln Sie ein Empfehlungssystem:
- Basierend auf Clustern: Welche Produkte sind ähnlich?
- Basierend auf Anomalien: Welche Qualitätsprobleme treten auf?
- Wie können diese Erkenntnisse die Produktion verbessern?
""")

print("\n🎯 ZUSAMMENFASSUNG MUSTERERKENNUNG")
print("=" * 40)

print("""
✅ GELERNTE KONZEPTE:
• K-Means Clustering für Produktgruppierung
• Optimale Cluster-Anzahl bestimmen
• Anomalieerkennung mit verschiedenen Methoden
• Bewertung von Clustering- und Anomalie-Algorithmen
• Datenvisualisierung für Mustererkennung

🔧 ANGEWENDETE METHODEN:
• K-Means Clustering
• Isolation Forest
• Statistische Anomalieerkennung (Z-Score)
• Principal Component Analysis (PCA)
• Silhouette-Analyse

💡 PRAKTISCHE ERKENNTNISSE:
• Datenvorverarbeitung ist entscheidend
• Verschiedene Algorithmen für verschiedene Muster
• Visualisierung hilft bei der Interpretation
• Bewertungsmetriken sind wichtig
• Domain-Wissen verbessert Ergebnisse

🏭 ANWENDUNG IN DER PRODUKTION:
• Produktgruppierung für Fertigungsplanung
• Qualitätskontrolle automatisieren
• Wartungsbedarfe vorhersagen
• Prozessoptimierung durch Mustererkennung
• Ausschuss reduzieren durch frühzeitige Erkennung
""")

print("\n📚 Nächster Schritt: KI-Integration & Zukunftsperspektiven!")